# EFFResNet-Unified: All-in-One Deepfake Detection Network
### Integrating Spatial CNNs, Frequency 2D-FFT, and SRM Steganalysis

**Unified Architecture Innovations:**
1. **Pretrained Baseline Backbone**: Initialized from our top-performing EFFResNet-ViT (warm start).
2. **2D-FFT Spectral Stream**: Detects periodic lattice upsampling artifacts in the frequency domain.
3. **5x5 SRM Steganalysis Stream**: Fixed steerable high-pass filters extracting face boundary blending residuals.
4. **Zero-Initialized Gated Cross-Attention**: $\tanh(\alpha)$ initialized at 0 guarantees zero baseline degradation.
5. **Multi-Paradigm Video-Level Voting**: Mean, Median, and Majority voting across video sequences.

In [ ]:
import os, sys, time, json, random
from pathlib import Path
from typing import List, Tuple, Dict
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import EfficientNet_B0_Weights, ResNet50_Weights

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 65)
print('PyTorch Version :', torch.__version__)
print('Device          :', device)
if torch.cuda.is_available():
    print('GPU Device      :', torch.cuda.get_device_name(0))
    print('VRAM Available  :', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), 'GB')
print('=' * 65)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

In [ ]:
# ── DATASET DISCOVERY & BALANCED SPLITTING ───────────────────────
def locate_dataset_root() -> Path:
    for p in [
        Path('/kaggle/input/faceforencispp-extracted-frames'),
        Path('/kaggle/input/datasets/adham7elmy/faceforencispp-extracted-frames'),
    ]:
        if p.exists() and (p / 'real').exists(): return p
    for sub in Path('/kaggle/input').rglob('real'):
        if sub.is_dir() and (sub.parent / 'fake').exists(): return sub.parent
    raise FileNotFoundError('FaceForensics++ dataset not found under /kaggle/input')

DATASET_ROOT = locate_dataset_root()
print('Dataset Root:', DATASET_ROOT)

def build_ffpp_manifest(root: Path, max_frames_per_video: int = 15, val_split: float = 0.2, seed: int = 42):
    random.seed(seed)
    real_dir = root / 'real'
    fake_dir = root / 'fake'
    
    real_videos = {}
    if real_dir.exists():
        for vid in sorted(real_dir.iterdir()):
            if vid.is_dir():
                frames = sorted([str(p) for p in vid.glob('*.png')])
                if frames: real_videos[vid.name] = frames
                
    fake_videos = {}
    if fake_dir.exists():
        for method in sorted(fake_dir.iterdir()):
            if method.is_dir():
                for vid in sorted(method.iterdir()):
                    if vid.is_dir():
                        frames = sorted([str(p) for p in vid.glob('*.png')])
                        if frames: fake_videos[f'{method.name}_{vid.name}'] = frames
                        
    real_keys = list(real_videos.keys())
    random.shuffle(real_keys)
    n_real_val = max(1, int(len(real_keys) * val_split))
    val_real_keys = set(real_keys[:n_real_val])
    train_real_keys = set(real_keys[n_real_val:])
    
    fake_keys = list(fake_videos.keys())
    random.shuffle(fake_keys)
    n_fake_val = max(1, int(len(fake_keys) * val_split))
    val_fake_keys = set(fake_keys[:n_fake_val])
    train_fake_keys = set(fake_keys[n_fake_val:])
    
    def sample_from_keys(keys, src, label):
        samples = []
        for k in keys:
            fl = src[k]
            chosen = fl if len(fl) <= max_frames_per_video else [
                fl[int(i * len(fl) / max_frames_per_video)] for i in range(max_frames_per_video)
            ]
            for f in chosen: samples.append((f, label, k))
        return samples
        
    tr_real = sample_from_keys(train_real_keys, real_videos, 0)
    va_real = sample_from_keys(val_real_keys, real_videos, 0)
    tr_fake = sample_from_keys(train_fake_keys, fake_videos, 1)
    va_fake = sample_from_keys(val_fake_keys, fake_videos, 1)
    
    n_tr = min(len(tr_real), len(tr_fake))
    train_samples = random.sample(tr_real, n_tr) + random.sample(tr_fake, n_tr)
    n_va = min(len(va_real), len(va_fake))
    val_samples = random.sample(va_real, n_va) + random.sample(va_fake, n_va)
    
    random.shuffle(train_samples)
    random.shuffle(val_samples)
    return train_samples, val_samples

train_samples, val_samples = build_ffpp_manifest(DATASET_ROOT, max_frames_per_video=15)
print(f'Train Samples : {len(train_samples):,} (Real: {len(train_samples)//2}, Fake: {len(train_samples)//2})')
print(f'Val Samples   : {len(val_samples):,} (Real: {len(val_samples)//2}, Fake: {len(val_samples)//2})')
print(f'Unique Val Videos: {len(set(e[2] for e in val_samples))}')

In [ ]:
# ── DATASETS & LOADERS ───────────────────────────────────────────
class FFPPUnifiedDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label, vid_id = self.samples[idx]
        try:    img = Image.open(path).convert('RGB')
        except: img = Image.new('RGB', (224, 224))
        if self.transform: img = self.transform(img)
        else: img = transforms.ToTensor()(img)
        return img, torch.tensor(label, dtype=torch.float32), vid_id

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_loader = DataLoader(FFPPUnifiedDataset(train_samples, train_tf), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(FFPPUnifiedDataset(val_samples, val_tf), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)}')

In [ ]:
# ── MODEL ARCHITECTURE: EFFRESNET-UNIFIED ────────────────────────
def get_srm_filters() -> torch.Tensor:
    f1 = np.array([
        [0,  0,  0,  0,  0],
        [0, -1,  2, -1,  0],
        [0,  2, -4,  2,  0],
        [0, -1,  2, -1,  0],
        [0,  0,  0,  0,  0]
    ], dtype=np.float32) / 4.0
    f2 = np.array([
        [-1,  2, -2,  2, -1],
        [ 2, -6,  8, -6,  2],
        [-2,  8,-12,  8, -2],
        [ 2, -6,  8, -6,  2],
        [-1,  2, -2,  2, -1]
    ], dtype=np.float32) / 12.0
    f3 = np.array([
        [0,  0,  0,  0,  0],
        [0,  1, -2,  1,  0],
        [0, -2,  4, -2,  0],
        [0,  1, -2,  1,  0],
        [0,  0,  0,  0,  0]
    ], dtype=np.float32) / 4.0
    filters = np.stack([f1, f2, f3], axis=0)
    return torch.tensor(np.expand_dims(filters, axis=1), dtype=torch.float32)

class SRMSteganalysisStream(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        srm_k = get_srm_filters()
        self.srm_conv = nn.Conv2d(3, 3, kernel_size=5, padding=2, groups=3, bias=False)
        self.srm_conv.weight.data = srm_k.repeat(1, 1, 1, 1)
        self.srm_conv.weight.requires_grad = False
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(256), nn.GELU(),
            nn.Conv2d(256, out_dim, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(out_dim), nn.GELU()
        )
    def forward(self, x):
        return self.encoder(self.srm_conv(x))

class SpectralFrequencyStream(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(256), nn.GELU(),
            nn.Conv2d(256, out_dim, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(out_dim), nn.GELU()
        )
    def forward(self, x):
        fft2d = torch.fft.fft2(x, norm='ortho')
        mag = torch.log(torch.abs(fft2d) + 1e-6)
        mag_shift = torch.fft.fftshift(mag, dim=(-2, -1))
        return self.encoder(mag_shift)

class ConvLocalityBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.act1 = nn.GELU()
        self.depthwise = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, groups=out_channels, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act2 = nn.GELU()
    def forward(self, x):
        return self.act2(self.bn2(self.depthwise(self.act1(self.bn1(self.conv1(x))))))

class CrossAttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(embed_dim)
        self.norm_kv = nn.LayerNorm(embed_dim)
        self.mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
    def forward(self, q, kv):
        q_n = self.norm_q(q)
        kv_n = self.norm_kv(kv)
        out, _ = self.mha(q_n, kv_n, kv_n)
        return self.dropout(out)

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        normed = self.norm1(x)
        attn_out, _ = self.mha(normed, normed, normed)
        x = x + self.dropout1(attn_out)
        x = x + self.mlp(self.norm2(x))
        return x

class EFFResNetUnified(nn.Module):
    def __init__(self, embed_dim=256, num_heads=8, num_vit_layers=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        eff = models.efficientnet_b0(weights=None)
        self.eff_features = eff.features
        res = models.resnet50(weights=None)
        self.res_stem = nn.Sequential(
            res.conv1, res.bn1, res.relu, res.maxpool,
            res.layer1, res.layer2, res.layer3, res.layer4
        )
        self.fusion_neck = ConvLocalityBlock(1280 + 2048, embed_dim)
        self.freq_stream = SpectralFrequencyStream(embed_dim)
        self.srm_stream = SRMSteganalysisStream(embed_dim)
        self.cross_attn_freq = CrossAttentionBlock(embed_dim, num_heads=num_heads, dropout=dropout)
        self.cross_attn_srm = CrossAttentionBlock(embed_dim, num_heads=num_heads, dropout=dropout)
        self.alpha_freq = nn.Parameter(torch.zeros(1))
        self.alpha_srm = nn.Parameter(torch.zeros(1))
        self.num_patches = 7 * 7
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.pos_dropout = nn.Dropout(dropout)
        self.transformer_layers = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(num_vit_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.post_attn_locality = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim, kernel_size=3, padding=1, groups=embed_dim, bias=False),
            nn.BatchNorm2d(embed_dim),
            nn.GELU()
        )
        self.feat_proj = nn.Linear(embed_dim * 2, embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        B = x.size(0)
        f_eff = self.eff_features(x)
        f_res = self.res_stem(x)
        if f_eff.shape[2:] != f_res.shape[2:]:
            f_eff = F.interpolate(f_eff, size=f_res.shape[2:], mode='bilinear', align_corners=False)
        f_spatial = self.fusion_neck(torch.cat([f_eff, f_res], dim=1))
        f_freq = self.freq_stream(x)
        f_srm = self.srm_stream(x)
        H, W = f_spatial.shape[2], f_spatial.shape[3]
        tokens_spatial = f_spatial.flatten(2).transpose(1, 2)
        tokens_freq = f_freq.flatten(2).transpose(1, 2)
        tokens_srm = f_srm.flatten(2).transpose(1, 2)
        freq_context = self.cross_attn_freq(tokens_spatial, tokens_freq)
        srm_context = self.cross_attn_srm(tokens_spatial, tokens_srm)
        tokens_fused = (
            tokens_spatial
            + torch.tanh(self.alpha_freq) * freq_context
            + torch.tanh(self.alpha_srm) * srm_context
        )
        cls_tokens = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls_tokens, tokens_fused], dim=1)
        tokens = self.pos_dropout(tokens + self.pos_embed[:, :tokens.size(1), :])
        for layer in self.transformer_layers: tokens = layer(tokens)
        tokens = self.norm(tokens)
        cls_out = tokens[:, 0]
        spatial_tokens = tokens[:, 1:]
        spatial_grid = spatial_tokens.transpose(1, 2).reshape(B, self.embed_dim, H, W)
        spatial_recovered = self.post_attn_locality(spatial_grid)
        spatial_pool = F.adaptive_avg_pool2d(spatial_recovered, (1, 1)).flatten(1)
        feat = self.feat_proj(torch.cat([cls_out, spatial_pool], dim=1))
        return self.classifier(feat)

model = EFFResNetUnified(embed_dim=256, num_heads=8, num_vit_layers=4).to(device)
print(f'Total Unified Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} M')

In [ ]:
# ── WARM-START TRANSFER FROM BASELINE CHECKPOINT ─────────────────
def find_baseline_ckpt() -> Path:
    for p in Path('/kaggle/input').rglob('best_effresnet_vit.pth'):
        if p.is_file(): return p
    for p in Path('.').rglob('best_effresnet_vit.pth'):
        if p.is_file(): return p
    return None

BASE_CKPT = find_baseline_ckpt()
if BASE_CKPT:
    print('Loading Warm-Start Baseline Checkpoint from:', BASE_CKPT)
    try:
        ckpt = torch.load(BASE_CKPT, map_location=device, weights_only=False)
    except TypeError:
        ckpt = torch.load(BASE_CKPT, map_location=device)
    st = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    res = model.load_state_dict(st, strict=False)
    print(f'Successfully transferred {len(st)} baseline weights! (New forensic keys: {len(res.missing_keys)})')
else:
    print('No baseline checkpoint mounted; initializing backbones with ImageNet weights.')
    eff_pre = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    res_pre = models.resnet50(weights=ResNet50_Weights.DEFAULT)
    model.eff_features.load_state_dict(eff_pre.features.state_dict())
    model.res_stem.load_state_dict(nn.Sequential(
        res_pre.conv1, res_pre.bn1, res_pre.relu, res_pre.maxpool,
        res_pre.layer1, res_pre.layer2, res_pre.layer3, res_pre.layer4
    ).state_dict())
    print('ImageNet pretraining loaded.')

In [ ]:
# ── TRAINING SETUP: LOSS, OPTIMIZER & SCHEDULER ──────────────────
NUM_EPOCHS = 5
OUTPUT_DIR = Path('/kaggle/working/output_unified')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))
criterion = nn.BCEWithLogitsLoss()

def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    acc = accuracy_score(y_true, y_pred) * 100.0
    prec = precision_score(y_true, y_pred, zero_division=0) * 100.0
    rec = recall_score(y_true, y_pred, zero_division=0) * 100.0
    f1 = f1_score(y_true, y_pred, zero_division=0) * 100.0
    auc = roc_auc_score(y_true, y_prob) * 100.0 if len(set(y_true)) > 1 else 50.0
    cm = confusion_matrix(y_true, y_pred).tolist()
    return {'accuracy': round(acc, 2), 'precision': round(prec, 2), 'recall': round(rec, 2), 'f1_score': round(f1, 2), 'auc_roc': round(auc, 2), 'confusion_matrix': cm}

In [ ]:
# ── TRAINING LOOP WITH CONTINUOUS VIDEO VOTING EVALUATION ────────
history = {
    'train_loss': [], 'val_loss': [],
    'frame_acc': [], 'frame_auc': [], 'frame_f1': [],
    'video_mean_acc': [], 'video_mean_auc': [], 'video_mean_f1': [],
    'alpha_freq': [], 'alpha_srm': []
}
best_video_acc = 0.0
best_checkpoint_path = OUTPUT_DIR / 'best_effresnet_unified.pth'

print('=' * 80)
print(f'Starting Training EFFResNet-Unified for {NUM_EPOCHS} Epochs')
print('=' * 80)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    t0 = time.time()

    for step, (imgs, lbls, _) in enumerate(train_loader):
        imgs = imgs.to(device, non_blocking=True)
        lbls_t = lbls.to(device, non_blocking=True).unsqueeze(1)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(imgs)
            loss = criterion(logits, lbls_t)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_train_loss += loss.item()
        if (step + 1) % max(1, len(train_loader) // 4) == 0:
            print(f'Epoch [{epoch}/{NUM_EPOCHS}] Step [{step+1}/{len(train_loader)}] Loss: {loss.item():.4f}')

    scheduler.step()
    avg_train_loss = total_train_loss / len(train_loader)

    # ── VALIDATION: DUAL FRAME & VIDEO VOTING ──
    model.eval()
    val_loss = 0.0
    all_lbls, all_probs, all_vids = [], [], []
    with torch.no_grad():
        for imgs, lbls, vids in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            lbls_t = lbls.to(device, non_blocking=True).unsqueeze(1)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                logits = model(imgs)
                val_loss += criterion(logits, lbls_t).item()
                probs = torch.sigmoid(logits)
            all_lbls.extend(lbls.numpy().tolist())
            all_probs.extend(probs.squeeze(1).cpu().numpy().tolist())
            all_vids.extend(vids)

    avg_val_loss = val_loss / len(val_loader)
    g_freq = torch.tanh(model.alpha_freq).item()
    g_srm = torch.tanh(model.alpha_srm).item()

    # Frame-Level Metrics
    m_frame = compute_metrics(np.array(all_lbls), np.array(all_probs))

    # Video-Level Aggregation
    vid_probs = defaultdict(list)
    vid_labels = {}
    for p, y, vid in zip(all_probs, all_lbls, all_vids):
        vid_probs[vid].append(p)
        vid_labels[vid] = y
    sorted_vids = sorted(vid_probs.keys())
    v_true = np.array([vid_labels[v] for v in sorted_vids])
    v_mean_prob = np.array([np.mean(vid_probs[v]) for v in sorted_vids])
    m_video = compute_metrics(v_true, v_mean_prob)

    elapsed = time.time() - t0
    history['train_loss'].append(round(avg_train_loss, 4))
    history['val_loss'].append(round(avg_val_loss, 4))
    history['frame_acc'].append(m_frame['accuracy'])
    history['frame_auc'].append(m_frame['auc_roc'])
    history['frame_f1'].append(m_frame['f1_score'])
    history['video_mean_acc'].append(m_video['accuracy'])
    history['video_mean_auc'].append(m_video['auc_roc'])
    history['video_mean_f1'].append(m_video['f1_score'])
    history['alpha_freq'].append(round(g_freq, 4))
    history['alpha_srm'].append(round(g_srm, 4))

    print(f'\nEpoch {epoch}/{NUM_EPOCHS} Summary ({elapsed:.1f}s):')
    print(f'  Train Loss     : {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')
    print(f'  Frame Metrics  : Acc = {m_frame["accuracy"]}% | AUC = {m_frame["auc_roc"]}% | F1 = {m_frame["f1_score"]}%')
    print(f'  Video Voting   : Acc = {m_video["accuracy"]}% | AUC = {m_video["auc_roc"]}% | Prec = {m_video["precision"]}%')
    print(f'  Gates Actives  : tanh(alpha_freq) = {g_freq:.4f} | tanh(alpha_srm) = {g_srm:.4f}')

    if m_video['accuracy'] > best_video_acc:
        best_video_acc = m_video['accuracy']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'frame_metrics': m_frame,
            'video_metrics': m_video,
            'history': history
        }, best_checkpoint_path)
        print(f'  ==> [BEST CHECKPOINT SAVED] Video Accuracy: {best_video_acc}%')

In [ ]:
# ── METRICS EXPORT & TRAINING VISUALIZATION ──────────────────────
with open(OUTPUT_DIR / 'metrics_unified.json', 'w') as f:
    json.dump(history, f, indent=2)

epochs_range = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'], 'r--s', label='Val Loss')
axes[0].set_title('Cross-Entropy Loss Progression')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs_range, history['frame_acc'], 'm-^', label='Frame Accuracy')
axes[1].plot(epochs_range, history['video_mean_acc'], 'g-o', linewidth=2, label='Video Voting Accuracy')
axes[1].set_title('Accuracy: Single-Frame vs Video Voting')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True)

axes[2].plot(epochs_range, history['alpha_freq'], 'b-d', label='Spectral Gate tanh(alpha_freq)')
axes[2].plot(epochs_range, history['alpha_srm'], 'orange', marker='s', label='Steganalysis Gate tanh(alpha_srm)')
axes[2].set_title('Learned Novelty Gating Weights')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Gate Amplitude')
axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves_unified.png', dpi=300)
plt.show()
print('Saved training summary and plots to', OUTPUT_DIR)